# LeetCode 1743 — Restore the Array From Adjacent Pairs bằng Python

Link đề: https://leetcode.com/problems/restore-the-array-from-adjacent-pairs/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Depth-First Search (đồ thị đường đi)

Skeleton LeetCode:

```python
class Solution:
    def restoreArray(self, adjacentPairs: List[List[int]]) -> List[int]:
        ...
```

> Lưu ý: đề cho phép trả về **bất kỳ** mảng hợp lệ nào — thường có đúng 2 đáp án: mảng gốc và mảng **đảo ngược**. Vì vậy trong notebook này ta **không** so sánh bằng `==` với một đáp án cố định, mà dùng hàm `is_valid(...)` kiểm tra tính hợp lệ.

Lộ trình: **thử mọi hoán vị** (baseline) → **nối dây tham lam** quét danh sách cặp `O(n²)` → **danh sách kề + bắt đầu từ đầu mút bậc 1** `O(n)` → **DFS bằng ngăn xếp** (tránh đệ quy sâu) → mẹo **XOR hàng xóm** gọn bộ nhớ.

## 1. Mục tiêu học

1. Nhìn một danh sách "cặp kề nhau" như một **đồ thị vô hướng**, và nhận ra đồ thị đó là một **đường đi** (path) — không nhánh, không chu trình.
2. Dùng `defaultdict(list)` để dựng **danh sách kề** và dùng **bậc** (số hàng xóm) để tìm hai **đầu mút** (bậc 1).
3. Duyệt một đường đi bằng quy tắc "đi tới hàng xóm **không phải** đỉnh vừa đến" — không cần tập `visited`.
4. Biết vì sao DFS **đệ quy** có thể vỡ `RecursionError` với `n = 10⁵` trong Python, và cách chuyển sang vòng lặp.
5. Viết **hàm kiểm tra hợp lệ** cho bài có nhiều đáp án, và test ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Có một mảng số nguyên `nums` gồm `n` phần tử **đôi một khác nhau**, nhưng bạn đã quên nó. Bạn chỉ còn nhớ **mọi cặp phần tử kề nhau** trong `nums`.

Cho mảng 2 chiều `adjacentPairs` kích thước `n - 1`, mỗi `adjacentPairs[i] = [u_i, v_i]` nghĩa là `u_i` và `v_i` đứng cạnh nhau trong `nums`. Mỗi cặp kề `nums[i], nums[i+1]` đều xuất hiện trong `adjacentPairs` dưới dạng `[nums[i], nums[i+1]]` **hoặc** `[nums[i+1], nums[i]]`, và các cặp có thể nằm **theo thứ tự bất kỳ**.

Hãy trả về mảng gốc `nums`. Nếu có nhiều đáp án, trả về **bất kỳ** đáp án nào.

**Ví dụ 1:** `adjacentPairs = [[2,1],[3,4],[3,2]]` → `[1,2,3,4]` (đảo ngược `[4,3,2,1]` cũng đúng).

**Ví dụ 2:** `adjacentPairs = [[4,-2],[1,4],[-3,1]]` → `[-2,4,1,-3]` (hoặc `[-3,1,4,-2]`). Có thể có số âm.

**Ví dụ 3:** `adjacentPairs = [[100000,-100000]]` → `[100000,-100000]`.

**Ràng buộc:**
- `nums.length == n`, `adjacentPairs.length == n - 1`, `adjacentPairs[i].length == 2`
- `2 ≤ n ≤ 10⁵`
- `-10⁵ ≤ nums[i], u_i, v_i ≤ 10⁵`
- Luôn tồn tại một `nums` sinh ra đúng `adjacentPairs`.

In [1]:
import random
import time
import sys
from collections import defaultdict, Counter
from itertools import permutations
from typing import List

examples = [
    [[2, 1], [3, 4], [3, 2]],
    [[4, -2], [1, 4], [-3, 1]],
    [[100000, -100000]],
]
expected_one = [[1, 2, 3, 4], [-2, 4, 1, -3], [100000, -100000]]  # một đáp án mẫu của đề

def is_valid(pairs, ans):
    """ans hợp lệ <=> đủ n = len(pairs)+1 phần tử khác nhau và tập cặp kề của ans
    trùng (không tính thứ tự trong cặp) với tập cặp đề cho."""
    n = len(pairs) + 1
    if len(ans) != n or len(set(ans)) != n:
        return False
    want = Counter(frozenset(p) for p in pairs)
    got = Counter(frozenset((ans[i], ans[i + 1])) for i in range(n - 1))
    return want == got

for p, e in zip(examples, expected_one):
    print(p, "-> đáp án mẫu", e, "| hợp lệ:", is_valid(p, e), "| đảo ngược hợp lệ:", is_valid(p, e[::-1]))

[[2, 1], [3, 4], [3, 2]] -> đáp án mẫu [1, 2, 3, 4] | hợp lệ: True | đảo ngược hợp lệ: True
[[4, -2], [1, 4], [-3, 1]] -> đáp án mẫu [-2, 4, 1, -3] | hợp lệ: True | đảo ngược hợp lệ: True
[[100000, -100000]] -> đáp án mẫu [100000, -100000] | hợp lệ: True | đảo ngược hợp lệ: True


Hàm `is_valid` sẽ là "trọng tài" cho mọi test bên dưới. Nó dùng `frozenset` để coi `[2,1]` và `[1,2]` là **cùng một cặp**, và `Counter` để so sánh hai **đa tập** cặp.

## 3. Kiến thức nền

### 3.1 `defaultdict(list)` — danh sách kề

`defaultdict(list)` tự tạo list rỗng khi truy cập khóa mới, rất tiện để dựng đồ thị: với mỗi cặp `[a, b]` ta thêm `b` vào hàng xóm của `a` **và** `a` vào hàng xóm của `b` (đồ thị vô hướng).

In [2]:
g = defaultdict(list)
for a, b in [[2, 1], [3, 4], [3, 2]]:
    g[a].append(b)
    g[b].append(a)
for k in sorted(g):
    print(k, "->", g[k])

1 -> [2]
2 -> [1, 3]
3 -> [4, 2]
4 -> [3]


### 3.2 Bậc của đỉnh

**Bậc** = số hàng xóm = `len(g[x])`. Trong một đường đi, hai đầu mút có bậc 1, các đỉnh giữa có bậc 2. Ta có thể đếm bậc riêng bằng `Counter`.

In [3]:
deg = Counter()
for a, b in [[4, -2], [1, 4], [-3, 1]]:
    deg[a] += 1
    deg[b] += 1
print(deg)
print("đầu mút (bậc 1):", [x for x, d in deg.items() if d == 1])

Counter({4: 2, 1: 2, -2: 1, -3: 1})
đầu mút (bậc 1): [-2, -3]


### 3.3 `next(...)` với biểu thức sinh

`next(x for x in iterable if điều_kiện)` lấy **phần tử đầu tiên** thỏa điều kiện mà không phải viết vòng `for ... break`.

In [4]:
g2 = {5: [7], 7: [5, 9], 9: [7]}
start = next(x for x, nb in g2.items() if len(nb) == 1)
print("đỉnh bậc 1 đầu tiên:", start)

đỉnh bậc 1 đầu tiên: 5


### 3.4 Biểu thức điều kiện để "chọn hàng xóm còn lại"

Một đỉnh giữa có đúng hai hàng xóm `v[0], v[1]`. Nếu một trong hai là đỉnh ta vừa đi qua (`prev`), hàng xóm kia là đỉnh tiếp theo: `nxt = v[1] if v[0] == prev else v[0]`.

In [5]:
v = [3, 1]          # hàng xóm của 2
prev = 1            # vừa đi từ 1 sang 2
nxt = v[1] if v[0] == prev else v[0]
print("đi tiếp tới:", nxt)

đi tiếp tới: 3


### 3.5 Phép XOR `^` và tính chất `a ^ b ^ a == b`

XOR có tính giao hoán, kết hợp, `x ^ x = 0`, `x ^ 0 = x`. Nếu lưu `X = a ^ b` (XOR hai hàng xóm), biết một hàng xóm `a` thì hàng xóm kia là `X ^ a`. Python XOR được cả số âm (biểu diễn bù hai vô hạn), nên vẫn đúng với `-10⁵`.

In [6]:
a, b = -3, 4
X = a ^ b
print("X =", X, "| X ^ a =", X ^ a, "| X ^ b =", X ^ b)

X = -7 | X ^ a = 4 | X ^ b = -3


### 3.6 Giới hạn đệ quy của Python

Python mặc định giới hạn độ sâu đệ quy khoảng 1000 (`sys.getrecursionlimit()`). Một DFS đệ quy trên đường đi dài `10⁵` sẽ vượt giới hạn — ta sẽ thấy ở mục Lỗi thường gặp.

In [7]:
print("recursion limit mặc định:", sys.getrecursionlimit())

recursion limit mặc định: 1000


## 4. Ý tưởng nền tảng / trực giác

Coi mỗi số là một **đỉnh**, mỗi cặp là một **cạnh**. Vì các cặp đến từ một mảng thẳng `nums[0] - nums[1] - ... - nums[n-1]`, đồ thị chính là **một sợi dây**:

```
Ví dụ 1: cặp [2,1], [3,4], [3,2]
   1 — 2 — 3 — 4
bậc: 1   2   2   1
```

Tính bằng tay:
1. Liệt kê hàng xóm: `1:[2]`, `2:[1,3]`, `3:[4,2]`, `4:[3]`.
2. Đỉnh bậc 1: `1` và `4` → đó là hai đầu dây. Chọn `1` làm đầu.
3. Từ `1` chỉ có một đường: sang `2`. Đến `2`, hàng xóm là `1` (vừa đi qua) và `3` → sang `3`. Đến `3`, hàng xóm `4` và `2` (vừa qua) → sang `4`. Hết: `[1,2,3,4]`.

Nếu bắt đầu từ `4` ta được `[4,3,2,1]` — cũng là một đáp án hợp lệ. Đó là lý do "trả mảng hoặc mảng đảo đều đúng".

**Câu hỏi then chốt:** nếu bắt đầu ở đỉnh **giữa** (bậc 2) thì sao? Ta phải chọn đi trái hay phải, và sẽ chỉ khôi phục được nửa dây. Vì vậy **phải bắt đầu ở đầu mút**.

In [8]:
pairs = [[2, 1], [3, 4], [3, 2]]
g = defaultdict(list)
for a, b in pairs:
    g[a].append(b); g[b].append(a)
print({k: g[k] for k in sorted(g)})
print("bậc:", {k: len(g[k]) for k in sorted(g)})

{1: [2], 2: [1, 3], 3: [4, 2], 4: [3]}
bậc: {1: 1, 2: 2, 3: 2, 4: 1}


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng:** tập giá trị của `nums` chính là tập mọi số xuất hiện trong các cặp. Thử từng **hoán vị** của tập đó, hoán vị nào có tập cặp kề trùng với đề thì trả về.

Cách này hiển nhiên đúng (ta duyệt hết không gian đáp án) nên dùng làm **chuẩn đối chiếu** cho input nhỏ.

In [9]:
def restore_brute(pairs):
    vals = sorted({x for p in pairs for x in p})
    for perm in permutations(vals):
        if is_valid(pairs, list(perm)):
            return list(perm)
    return None

for p in examples:
    t = time.perf_counter()
    r = restore_brute(p)
    print(p, "->", r, "| hợp lệ:", is_valid(p, r), f"| {1000*(time.perf_counter()-t):.2f} ms")

[[2, 1], [3, 4], [3, 2]] -> [1, 2, 3, 4] | hợp lệ: True | 0.03 ms
[[4, -2], [1, 4], [-3, 1]] -> [-3, 1, 4, -2] | hợp lệ: True | 0.03 ms
[[100000, -100000]] -> [-100000, 100000] | hợp lệ: True | 0.01 ms


**Độ phức tạp:** có `n!` hoán vị, mỗi hoán vị kiểm tra mất `O(n)` ⇒ `O(n!·n)` thời gian, `O(n)` bộ nhớ.

**Nút thắt:** `n = 10` đã là 3,6 triệu hoán vị; `n = 10⁵` thì không tưởng. Ta lãng phí vì **không dùng thông tin cấu trúc**: phần lớn hoán vị vi phạm ngay ở cặp đầu tiên. Thử đo với `n = 8`:

In [10]:
rng = random.Random(1)
arr = rng.sample(range(-50, 50), 8)
pp = [[arr[i], arr[i + 1]] for i in range(7)]
t = time.perf_counter()
r = restore_brute(pp)
print("n=8:", r, "hợp lệ:", is_valid(pp, r), f"| {time.perf_counter()-t:.3f} s")

n=8: [-33, 22, 47, -42, -18, -35, 13, 7] hợp lệ: True | 0.074 s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — đồ thị là một đường đi.** Mỗi phần tử giữa mảng có đúng 2 hàng xóm, hai phần tử đầu/cuối có đúng 1 hàng xóm. Do các giá trị **khác nhau**, không có cạnh nào bị "gộp" — bậc phản ánh đúng vị trí.

**Insight 2 — đầu mút là chỗ bắt đầu duy nhất an toàn.** *Nếu* một đỉnh có bậc 1 *thì* nó là `nums[0]` hoặc `nums[n-1]`; bắt đầu ở đó thì ở mỗi bước chỉ có **một** lựa chọn đi tiếp.

**Insight 3 — quy tắc đi tiếp.** *Nếu* đang ở `cur` và vừa đến từ `prev` *thì* bước tiếp theo là hàng xóm của `cur` **khác** `prev`. Vì giá trị khác nhau nên "khác `prev`" xác định duy nhất, không cần mảng `visited`.

**Invariant khi duyệt:** sau `k` bước, `ans[0..k]` là một đoạn đầu (prefix) đúng của `nums` (hoặc của `nums` đảo ngược) và `prev = ans[k-1]`, `cur = ans[k]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nối dây tham lam, quét danh sách cặp (`O(n²)`)

Bước trung gian cho người mới chưa quen đồ thị: ta vẫn tìm đầu mút bằng cách đếm số lần xuất hiện, rồi **lặp lại**: quét danh sách cặp chưa dùng, tìm cặp chứa `cur`, nối đầu kia vào mảng, đánh dấu đã dùng.

In [11]:
def restore_scan(pairs):
    cnt = Counter(x for p in pairs for x in p)
    start = next(x for x, c in cnt.items() if c == 1)   # đầu mút xuất hiện đúng 1 lần
    used = [False] * len(pairs)
    ans = [start]
    cur = start
    for _ in range(len(pairs)):
        for i, (a, b) in enumerate(pairs):
            if not used[i] and (a == cur or b == cur):
                used[i] = True
                cur = b if a == cur else a
                ans.append(cur)
                break
    return ans

for p in examples:
    r = restore_scan(p)
    print(p, "->", r, "| hợp lệ:", is_valid(p, r))

[[2, 1], [3, 4], [3, 2]] -> [1, 2, 3, 4] | hợp lệ: True
[[4, -2], [1, 4], [-3, 1]] -> [-2, 4, 1, -3] | hợp lệ: True
[[100000, -100000]] -> [100000, -100000] | hợp lệ: True


Vì sao cặp chứa `cur` mà **chưa dùng** là duy nhất? `cur` có tối đa 2 cạnh, một cạnh (nối với `prev`) đã dùng, nên chỉ còn một.

**Độ phức tạp:** mỗi bước quét lại tới `n` cặp ⇒ `O(n²)` thời gian, `O(n)` bộ nhớ. Với `n = 10⁵` là `10¹⁰` phép — quá chậm. Nút thắt: **tìm cặp chứa `cur`** mất `O(n)`; ta cần tra cứu `O(1)` ⇒ **bảng băm hàng xóm**.

### 7.2 Cách 3 — Danh sách kề + đi từ đầu mút (`O(n)`) — lời giải chuẩn

Ba đoạn:

1. **Dựng đồ thị:** `g[a].append(b); g[b].append(a)` cho mỗi cặp.
2. **Tìm đầu mút:** đỉnh có `len(g[x]) == 1`. Đặt `ans[0] = x`, `ans[1] = g[x][0]`.
3. **Đi tiếp:** với `i ≥ 2`, `v = g[ans[i-1]]` có 2 phần tử; `ans[i]` là phần tử khác `ans[i-2]`.

Code demo có in từng bước:

In [12]:
def restore_adj_demo(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
    n = len(pairs) + 1
    ans = [0] * n
    start = next(x for x, v in g.items() if len(v) == 1)
    ans[0], ans[1] = start, g[start][0]
    print(f"bắt đầu ở đầu mút {start}, bước 1 -> {ans[1]}")
    for i in range(2, n):
        v = g[ans[i - 1]]
        ans[i] = v[1] if v[0] == ans[i - 2] else v[0]
        print(f"i={i}: tại {ans[i-1]} hàng xóm {v}, vừa đến từ {ans[i-2]} -> đi {ans[i]}")
    return ans

print(restore_adj_demo([[4, -2], [1, 4], [-3, 1]]))

bắt đầu ở đầu mút -2, bước 1 -> 4
i=2: tại 4 hàng xóm [-2, 1], vừa đến từ -2 -> đi 1
i=3: tại 1 hàng xóm [4, -3], vừa đến từ 4 -> đi -3
[-2, 4, 1, -3]


Code nộp (không in):

In [13]:
class SolutionAdj:
    def restoreArray(self, adjacentPairs: List[List[int]]) -> List[int]:
        g = defaultdict(list)
        for a, b in adjacentPairs:
            g[a].append(b)
            g[b].append(a)
        n = len(adjacentPairs) + 1
        ans = [0] * n
        for x, v in g.items():
            if len(v) == 1:
                ans[0], ans[1] = x, v[0]
                break
        for i in range(2, n):
            v = g[ans[i - 1]]
            ans[i] = v[1] if v[0] == ans[i - 2] else v[0]
        return ans

for p in examples:
    r = SolutionAdj().restoreArray(p)
    print(r, is_valid(p, r))

[1, 2, 3, 4] True
[-2, 4, 1, -3] True
[100000, -100000] True


**Độ phức tạp:** dựng đồ thị `O(n)`, tìm đầu mút `O(n)`, đi `O(n)` ⇒ **`O(n)` thời gian**, `O(n)` bộ nhớ (bảng băm `n` khóa, tổng `2(n-1)` phần tử trong các list).

### 7.3 Cách 4 — DFS bằng ngăn xếp tường minh

DFS là cách "tổng quát" để duyệt đồ thị. Trên đường đi, DFS từ đầu mút cho đúng thứ tự mảng. Để tránh giới hạn đệ quy, ta dùng **ngăn xếp** chứa cặp `(đỉnh, cha)`.

In [14]:
class SolutionStack:
    def restoreArray(self, adjacentPairs: List[List[int]]) -> List[int]:
        g = defaultdict(list)
        for a, b in adjacentPairs:
            g[a].append(b)
            g[b].append(a)
        start = next(x for x, v in g.items() if len(v) == 1)
        ans = []
        stack = [(start, None)]          # None: không trùng với giá trị nào trong đề
        while stack:
            node, parent = stack.pop()
            ans.append(node)
            for nb_ in g[node]:
                if nb_ != parent:
                    stack.append((nb_, node))
        return ans

for p in examples:
    r = SolutionStack().restoreArray(p)
    print(r, is_valid(p, r))

[1, 2, 3, 4] True
[-2, 4, 1, -3] True
[100000, -100000] True


Lưu ý chọn "cha giả" cho đỉnh đầu: dùng `None` (hoặc số ngoài miền `[-10⁵, 10⁵]`). **Không** dùng `0` hay `-1` vì đó là giá trị hợp lệ trong đề (xem Lỗi 3).

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ. Trên đường đi, ngăn xếp luôn chỉ có tối đa 1 phần tử chờ, nhưng code tổng quát vẫn đúng cho cây.

### 7.4 Cách 5 — Mẹo XOR hàng xóm (không cần list)

Mỗi đỉnh có tối đa 2 hàng xóm. Thay vì lưu list, lưu `xr[x] = XOR các hàng xóm` và `deg[x]`. Khi đi từ `prev` tới `cur`, hàng xóm tiếp theo là `xr[cur] ^ prev`. Ở đầu mút bậc 1, `xr[x]` chính là hàng xóm duy nhất.

In [15]:
class SolutionXor:
    def restoreArray(self, adjacentPairs: List[List[int]]) -> List[int]:
        xr = defaultdict(int)
        deg = defaultdict(int)
        for a, b in adjacentPairs:
            xr[a] ^= b; xr[b] ^= a
            deg[a] += 1; deg[b] += 1
        start = next(x for x, d in deg.items() if d == 1)
        n = len(adjacentPairs) + 1
        ans = [start, xr[start]]
        for _ in range(n - 2):
            prev, cur = ans[-2], ans[-1]
            ans.append(xr[cur] ^ prev)
        return ans

for p in examples:
    r = SolutionXor().restoreArray(p)
    print(r, is_valid(p, r))

[1, 2, 3, 4] True
[-2, 4, 1, -3] True
[100000, -100000] True


**Độ phức tạp:** vẫn `O(n)` thời gian và `O(n)` bộ nhớ, nhưng mỗi khóa chỉ lưu 2 số nguyên thay vì một list — hằng số bộ nhớ nhỏ hơn. Đây là mẹo hay để "khoe" trong phỏng vấn, nhưng Cách 3 dễ đọc hơn.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2: `adjacentPairs = [[4,-2],[1,4],[-3,1]]`.

**Dựng đồ thị:**

| cặp | `g` sau khi thêm |
|---|---|
| `[4,-2]` | `4:[-2]`, `-2:[4]` |
| `[1,4]` | `4:[-2,1]`, `1:[4]` |
| `[-3,1]` | `1:[4,-3]`, `-3:[1]` |

Đỉnh bậc 1 đầu tiên khi duyệt dict (theo thứ tự chèn): `-2`.

| `i` | `ans[i-2]` | `ans[i-1]` | `g[ans[i-1]]` | `ans[i]` | `ans` |
|---|---|---|---|---|---|
| 0,1 | — | — | — | `-2`, `4` | `[-2,4]` |
| 2 | -2 | 4 | `[-2,1]` | 1 | `[-2,4,1]` |
| 3 | 4 | 1 | `[4,-3]` | -3 | `[-2,4,1,-3]` |

In [16]:
def trace_restore(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
        print(f"thêm cạnh {a}—{b}: g = {dict(g)}")
    n = len(pairs) + 1
    start = next(x for x, v in g.items() if len(v) == 1)
    ans = [start, g[start][0]]
    print(f"đầu mút = {start}; ans = {ans}")
    for i in range(2, n):
        v = g[ans[i - 1]]
        nxt = v[1] if v[0] == ans[i - 2] else v[0]
        ans.append(nxt)
        print(f"i={i}: prev={ans[i-2]:>3} cur={ans[i-1]:>3} hàng xóm={v} -> next={nxt:>3} | ans={ans}")
    return ans

trace_restore([[4, -2], [1, 4], [-3, 1]])

thêm cạnh 4—-2: g = {4: [-2], -2: [4]}
thêm cạnh 1—4: g = {4: [-2, 1], -2: [4], 1: [4]}
thêm cạnh -3—1: g = {4: [-2, 1], -2: [4], 1: [4, -3], -3: [1]}
đầu mút = -2; ans = [-2, 4]
i=2: prev= -2 cur=  4 hàng xóm=[-2, 1] -> next=  1 | ans=[-2, 4, 1]
i=3: prev=  4 cur=  1 hàng xóm=[4, -3] -> next= -3 | ans=[-2, 4, 1, -3]


[-2, 4, 1, -3]

Trace Cách 5 (XOR) trên Ví dụ 1 để thấy `xr[cur] ^ prev` "gỡ" đúng hàng xóm còn lại:

In [17]:
def trace_xor(pairs):
    xr = defaultdict(int); deg = defaultdict(int)
    for a, b in pairs:
        xr[a] ^= b; xr[b] ^= a; deg[a] += 1; deg[b] += 1
    print("xr =", dict(xr), "| deg =", dict(deg))
    start = next(x for x, d in deg.items() if d == 1)
    ans = [start, xr[start]]
    print(f"bắt đầu {start}, hàng xóm duy nhất xr[{start}]={xr[start]}")
    while len(ans) < len(pairs) + 1:
        prev, cur = ans[-2], ans[-1]
        nxt = xr[cur] ^ prev
        print(f"cur={cur}, prev={prev}: xr[cur]={xr[cur]} ^ {prev} = {nxt}")
        ans.append(nxt)
    return ans

print(trace_xor([[2, 1], [3, 4], [3, 2]]))

xr = {2: 2, 1: 2, 3: 6, 4: 3} | deg = {2: 2, 1: 1, 3: 2, 4: 1}
bắt đầu 1, hàng xóm duy nhất xr[1]=2
cur=2, prev=1: xr[cur]=2 ^ 1 = 3
cur=3, prev=2: xr[cur]=6 ^ 2 = 4
[1, 2, 3, 4]


## 9. Code nộp LeetCode

Bản chuẩn (Cách 3). LeetCode đã `from typing import List` và `collections` sẵn, nhưng import tường minh không hại gì.

In [18]:
from collections import defaultdict
from typing import List

class Solution:
    def restoreArray(self, adjacentPairs: List[List[int]]) -> List[int]:
        g = defaultdict(list)
        for a, b in adjacentPairs:
            g[a].append(b)
            g[b].append(a)
        n = len(adjacentPairs) + 1
        ans = [0] * n
        for x, v in g.items():
            if len(v) == 1:          # đầu mút của đường đi
                ans[0], ans[1] = x, v[0]
                break
        for i in range(2, n):
            v = g[ans[i - 1]]        # đỉnh giữa: đúng 2 hàng xóm
            ans[i] = v[1] if v[0] == ans[i - 2] else v[0]
        return ans

for p in examples:
    r = Solution().restoreArray(p)
    assert is_valid(p, r)
    print(r)

[1, 2, 3, 4]
[-2, 4, 1, -3]
[100000, -100000]


## 10. Vì sao thuật toán đúng

Gọi `nums = [x₀, x₁, …, x_{n-1}]` là một mảng gốc hợp lệ.

1. **Bậc:** mỗi cặp `(x_i, x_{i+1})` đóng góp 1 vào bậc của cả hai. `x₀` và `x_{n-1}` chỉ thuộc 1 cặp ⇒ bậc 1; `x_i` ở giữa thuộc 2 cặp ⇒ bậc 2. Vì các giá trị khác nhau, không có hai vị trí cùng một đỉnh. Vậy đỉnh bậc 1 **chỉ có thể** là `x₀` hoặc `x_{n-1}`.
2. Không mất tổng quát giả sử ta bắt đầu ở `x₀` (nếu bắt đầu ở `x_{n-1}` thì lập luận y hệt cho `nums` đảo ngược — cũng là đáp án hợp lệ). `x₀` có hàng xóm duy nhất `x₁` ⇒ `ans[1] = x₁`.
3. **Quy nạp:** giả sử `ans[0..i-1] = x₀..x_{i-1}`. Hàng xóm của `x_{i-1}` (với `i-1` ở giữa) là đúng `{x_{i-2}, x_i}`. Ta chọn phần tử khác `ans[i-2] = x_{i-2}` ⇒ `ans[i] = x_i`.
4. Kết thúc sau `n` phần tử ⇒ `ans = nums`.

Với Cách 5: nếu hàng xóm của `cur` là `{prev, nxt}` thì `xr[cur] = prev ^ nxt` nên `xr[cur] ^ prev = nxt` — đúng cùng quy tắc ở bước 3.

## 11. Phân tích Time / Space

Gọi `n` = số phần tử mảng (= số cặp + 1).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Hoán vị | `O(n!·n)` | `O(n)` | chỉ dùng `n ≤ 8` |
| 2. Quét cặp | `O(n²)` | `O(n)` | mỗi bước quét lại danh sách |
| 3. Danh sách kề | `O(n)` trung bình | `O(n)` | tra bảng băm `O(1)` trung bình |
| 4. DFS ngăn xếp | `O(n)` | `O(n)` | tổng quát, không đệ quy |
| 5. XOR | `O(n)` | `O(n)` (hằng số nhỏ) | mỗi khóa 2 số nguyên |

"Trung bình" vì thao tác dict của Python là `O(1)` trung bình; worst-case lý thuyết của bảng băm là `O(n)` mỗi thao tác nhưng thực tế với số nguyên nhỏ không xảy ra. Đo thực nghiệm với `n = 10⁵`:

In [19]:
def gen_case(n, rng):
    arr = rng.sample(range(-10**5, 10**5 + 1), n)
    pairs = [[arr[i], arr[i + 1]] if rng.random() < 0.5 else [arr[i + 1], arr[i]] for i in range(n - 1)]
    rng.shuffle(pairs)
    return arr, pairs

rng = random.Random(2024)
arr, big = gen_case(10**5, rng)
for name, sol in [("Cách 3", SolutionAdj()), ("Cách 4", SolutionStack()), ("Cách 5", SolutionXor())]:
    t = time.perf_counter()
    r = sol.restoreArray(big)
    dt = time.perf_counter() - t
    print(f"{name}: hợp lệ={is_valid(big, r)}, khớp gốc hoặc đảo={r == arr or r == arr[::-1]}, {dt*1000:.1f} ms")

Cách 3: hợp lệ=True, khớp gốc hoặc đảo=True, 82.9 ms


Cách 4: hợp lệ=True, khớp gốc hoặc đảo=True, 98.5 ms


Cách 5: hợp lệ=True, khớp gốc hoặc đảo=True, 124.2 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

- `n = 2` (một cặp): cả hai đỉnh đều bậc 1.
- Có số `0` và số âm (bẫy "cha giả").
- Giá trị biên `±10⁵`.
- Cặp bị lật chiều và xáo thứ tự.

In [20]:
sols = [Solution(), SolutionAdj(), SolutionStack(), SolutionXor()]
edge = examples + [
    [[0, 5]],                         # n = 2, có 0
    [[5, 0], [0, 7]],                 # 0 ở giữa
    [[0, -1], [-1, 1]],               # 0 là đầu mút
    [[-100000, 100000], [100000, 0], [0, 99999]],
    [[3, 1], [2, 3], [4, 2], [5, 4]], # lật chiều mọi cặp
]
for p in edge:
    outs = [s.restoreArray(p) for s in sols]
    assert all(is_valid(p, o) for o in outs), (p, outs)
    assert is_valid(p, restore_scan(p))
    print("OK", p, "->", outs[0])

OK [[2, 1], [3, 4], [3, 2]] -> [1, 2, 3, 4]
OK [[4, -2], [1, 4], [-3, 1]] -> [-2, 4, 1, -3]
OK [[100000, -100000]] -> [100000, -100000]
OK [[0, 5]] -> [0, 5]
OK [[5, 0], [0, 7]] -> [5, 0, 7]
OK [[0, -1], [-1, 1]] -> [0, -1, 1]
OK [[-100000, 100000], [100000, 0], [0, 99999]] -> [-100000, 100000, 0, 99999]
OK [[3, 1], [2, 3], [4, 2], [5, 4]] -> [1, 3, 2, 4, 5]


### 12.2 Ngẫu nhiên có seed, đối chiếu brute force (n nhỏ) và nhau (n lớn)

Brute force trả một đáp án cụ thể; các cách khác có thể trả bản đảo ngược. Vì vậy ta kiểm tra: kết quả `r` hợp lệ **và** `r` bằng đáp án brute **hoặc** bản đảo của nó (đối với đường đi chỉ có đúng 2 đáp án).

In [21]:
rng = random.Random(7)
cnt = 0
for _ in range(300):
    n = rng.randint(2, 6)
    arr, pairs = gen_case(n, rng) if rng.random() < 0.5 else (None, None)
    if pairs is None:
        arr = rng.sample(range(-5, 6), n)          # miền nhỏ, hay chứa 0 và số âm
        pairs = [[arr[i], arr[i + 1]] for i in range(n - 1)]
        rng.shuffle(pairs)
    b = restore_brute(pairs)
    for s in sols:
        r = s.restoreArray(pairs)
        assert is_valid(pairs, r)
        assert r == b or r == b[::-1]
    assert is_valid(pairs, restore_scan(pairs))
    cnt += 1
print("đã kiểm", cnt, "case nhỏ")

for _ in range(30):
    n = rng.randint(2, 3000)
    arr, pairs = gen_case(n, rng)
    for s in sols:
        r = s.restoreArray(pairs)
        assert is_valid(pairs, r) and (r == arr or r == arr[::-1])
print("đã kiểm 30 case lớn")

đã kiểm 300 case nhỏ


đã kiểm 30 case lớn


### 12.3 Kiểm chứng tính chất "chỉ có đúng 2 đáp án"

Với `n ≥ 2` và giá trị khác nhau, brute force liệt kê **mọi** hoán vị hợp lệ — phải có đúng 2 (mảng và bản đảo).

In [22]:
def all_valid(pairs):
    vals = sorted({x for p in pairs for x in p})
    return [list(q) for q in permutations(vals) if is_valid(pairs, list(q))]

rng = random.Random(11)
for _ in range(50):
    n = rng.randint(2, 6)
    arr = rng.sample(range(-9, 10), n)
    pairs = [[arr[i], arr[i + 1]] for i in range(n - 1)]
    sols_all = all_valid(pairs)
    assert len(sols_all) == 2 and arr in sols_all and arr[::-1] in sols_all
print("mọi case: đúng 2 đáp án (mảng và mảng đảo)")

mọi case: đúng 2 đáp án (mảng và mảng đảo)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Bắt đầu từ một đỉnh bất kỳ (ví dụ `adjacentPairs[0][0]`)

In [23]:
def wrong_start(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
    n = len(pairs) + 1
    ans = [pairs[0][0], g[pairs[0][0]][0]]      # SAI: pairs[0][0] chưa chắc là đầu mút
    while len(ans) < n:
        v = g[ans[-1]]
        if len(v) == 1:                          # đụng đầu dây sớm -> dừng
            break
        ans.append(v[1] if v[0] == ans[-2] else v[0])
    return ans

p = [[2, 1], [3, 4], [3, 2]]
r = wrong_start(p)
print("kết quả:", r, "| hợp lệ:", is_valid(p, r))

kết quả: [2, 1] | hợp lệ: False


**Vì sao sai:** `2` nằm giữa dây; đi từ `2` sang `1` thì đụng đầu mút ngay, phần `3, 4` bị bỏ sót. **Sửa:** luôn chọn đỉnh có `len(g[x]) == 1`.

In [24]:
r = Solution().restoreArray(p)
print("đúng:", r, is_valid(p, r))

đúng: [1, 2, 3, 4] True


### Lỗi 2 — DFS đệ quy trên `n = 10⁵`

Code DFS đệ quy trông rất gọn, nhưng độ sâu đệ quy bằng độ dài mảng. Ta chạy thử với `n = 5000` (vượt giới hạn 1000) và **bắt** lỗi để thấy rõ:

In [25]:
def restore_recursive(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
    ans = []
    def dfs(u, parent):
        ans.append(u)
        for v in g[u]:
            if v != parent:
                dfs(v, u)
    dfs(next(x for x, v in g.items() if len(v) == 1), None)
    return ans

arr5k, p5k = gen_case(5000, random.Random(3))
try:
    restore_recursive(p5k)
    print("không lỗi (recursion limit đã được nâng?)")
except RecursionError as e:
    print("RecursionError:", str(e)[:60])

RecursionError: maximum recursion depth exceeded


**Vì sao sai:** mỗi lời gọi `dfs` chiếm một khung stack; 5000 (hay 10⁵) khung vượt `sys.getrecursionlimit()`. Nâng `sys.setrecursionlimit` là cách chữa cháy có thể làm sập tiến trình (tràn stack C). **Sửa:** dùng vòng lặp (Cách 3) hoặc ngăn xếp (Cách 4).

In [26]:
r = SolutionStack().restoreArray(p5k)
print("ngăn xếp tường minh:", is_valid(p5k, r))

ngăn xếp tường minh: True


### Lỗi 3 — Dùng giá trị hợp lệ làm "cha giả"

Nhiều bạn viết `dfs(start, 0)` hoặc `parent = -1`. Nhưng `0` và `-1` hoàn toàn có thể nằm trong `nums`!

In [27]:
def wrong_sentinel(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
    start = next(x for x, v in g.items() if len(v) == 1)
    ans = []
    stack = [(start, 0)]          # SAI: 0 có thể là một giá trị thật
    while stack:
        u, par = stack.pop()
        ans.append(u)
        for v in g[u]:
            if v != par:
                stack.append((v, u))
    return ans

p = [[5, 0], [0, 7]]
r = wrong_sentinel(p)
print("kết quả:", r, "| hợp lệ:", is_valid(p, r))

kết quả: [5] | hợp lệ: False


**Vì sao sai:** bắt đầu ở `5` với cha giả `0`; hàng xóm duy nhất của `5` là `0` — trùng "cha" nên bị bỏ qua, dừng ngay với `[5]`. **Sửa:** dùng `None` hoặc số ngoài miền giá trị (ví dụ `10**6`, như nhiều lời giải dùng `1e6`).

In [28]:
r = SolutionStack().restoreArray(p)
print("đúng:", r, is_valid(p, r))

đúng: [5, 0, 7] True


### Lỗi 4 — Lưu hàng xóm bằng `g[a] = b` (ghi đè)

In [29]:
def wrong_overwrite(pairs):
    g = {}
    for a, b in pairs:
        g[a] = b; g[b] = a       # SAI: đỉnh giữa có 2 hàng xóm, cái sau đè cái trước
    cnt = Counter(x for q in pairs for x in q)
    start = next(x for x, c in cnt.items() if c == 1)
    ans = [start]
    seen = {start}
    while len(ans) < len(pairs) + 1:
        nxt = g[ans[-1]]
        if nxt in seen:
            break
        ans.append(nxt); seen.add(nxt)
    return ans

p = [[2, 1], [3, 4], [3, 2]]
r = wrong_overwrite(p)
print("g bị ghi đè; kết quả:", r, "| hợp lệ:", is_valid(p, r))

g bị ghi đè; kết quả: [1, 2, 3] | hợp lệ: False


**Vì sao sai:** sau cặp `[3,2]`, `g[2] = 3` đè mất `g[2] = 1`, và `g[3] = 2` đè mất `g[3] = 4`. **Sửa:** lưu **list** hàng xóm (`defaultdict(list)`) hoặc XOR (Cách 5).

In [30]:
print("đúng:", Solution().restoreArray(p))

đúng: [1, 2, 3, 4]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Cho các cặp kề nhau / các cạnh, khôi phục thứ tự"** ⇒ dựng đồ thị, tìm đỉnh có **bậc đặc biệt** (bậc 1 cho đường đi, bậc vào 0 cho sắp xếp tô-pô).
- **Giá trị phân biệt** ⇒ quy tắc "đi tới hàng xóm khác `prev`" đủ thay `visited`.
- **Đồ thị là đường đi/cây** và `n` lớn ⇒ tránh DFS đệ quy trong Python; dùng vòng lặp.
- **Nhiều đáp án** ⇒ tự viết `is_valid` khi luyện tập; đừng so `==` với output mẫu.
- **Cha giả**: dùng `None` hoặc giá trị ngoài miền ràng buộc.
- Mẹo XOR: khi mỗi đỉnh có ≤ 2 hàng xóm, `xr[x]` thay được list (cũng là ý tưởng của *XOR linked list*).

## 15. Biến thể của bài

1. **Mảng vòng (circular):** cho `n` cặp cho mảng vòng `n` phần tử — mọi đỉnh bậc 2, không có đầu mút; bắt đầu ở đỉnh bất kỳ, chọn một trong hai hướng.
2. **Nhiều mảng rời nhau:** các cặp sinh từ nhiều mảng — đồ thị là **rừng các đường đi**; khôi phục từng thành phần.
3. **Giá trị có thể trùng:** khi đó bậc không còn phản ánh vị trí, bài thành tìm **đường đi Euler** (Hierholzer) — ví dụ LeetCode 332 *Reconstruct Itinerary*, 2097 *Valid Arrangement of Pairs*.
4. **Cặp có hướng** `[u, v]` nghĩa là `u` đứng **trước** `v`: tìm đỉnh có bậc vào 0 rồi đi theo cạnh ra (dạng tô-pô đơn giản).
5. **Kiểm tra** xem một tập cặp có sinh ra từ mảng nào không (có thể có chu trình/nhánh): kiểm bậc ≤ 2, đúng 2 đỉnh bậc 1, đồ thị liên thông.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi thứ tự, kiểm `is_valid` | `O(n!·n)` | `O(n)` | test chuẩn `n ≤ 8` |
| 2. Quét cặp | từ đầu mút, tìm cặp chứa `cur` bằng quét tuyến tính | `O(n²)` | `O(n)` | hiểu ý tưởng, `n` nhỏ |
| 3. Danh sách kề | `g` + đầu mút + "hàng xóm khác prev" | `O(n)` | `O(n)` | **nộp bài**, dễ đọc |
| 4. DFS ngăn xếp | DFS tổng quát không đệ quy | `O(n)` | `O(n)` | khi đã quen khung DFS/cây |
| 5. XOR | `xr[cur] ^ prev` | `O(n)` | `O(n)` hằng số nhỏ | tối ưu bộ nhớ, phỏng vấn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** Với `pairs = [[10, 20], [30, 10], [40, 30]]`, dict bậc là gì và hai đầu mút là số nào?

**Bài 2 (tính tay):** Chạy Cách 5 (XOR) bằng tay cho `pairs = [[7, -1], [0, 7]]`. Ghi `xr`, đỉnh bắt đầu (đỉnh bậc 1 đầu tiên theo thứ tự chèn) và mảng kết quả.

**Bài 3 (điền TODO):** Viết `find_endpoints(pairs)` trả về **bộ đã sắp xếp** hai đầu mút.

**Bài 4 (tự viết lại):** Viết lại lời giải `O(n)` **không dùng** `defaultdict` — chỉ dùng `dict` thường với `setdefault`.

**Bài 5 (mini challenge — mảng vòng):** Cho `pairs` gồm `n` cặp sinh từ một **mảng vòng** `n ≥ 3` phần tử khác nhau (phần tử cuối kề phần tử đầu). Trả về mảng **chuẩn hóa**: bắt đầu từ phần tử nhỏ nhất, phần tử thứ hai là hàng xóm **nhỏ hơn** trong hai hàng xóm của nó.

In [31]:
def find_endpoints(pairs):
    # TODO: đếm bậc, trả về tuple(sorted([u, v])) của hai đỉnh bậc 1
    pass

def restore_plain_dict(pairs):
    # TODO: như Cách 3 nhưng dùng dict thường + setdefault
    pass

def restore_cycle(pairs):
    # TODO: mảng vòng, chuẩn hóa bắt đầu từ min, đi về phía hàng xóm nhỏ hơn
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 (nhẹ) | Gợi ý 2 (vừa) | Gợi ý 3 (gần lời giải) |
|---|---|---|---|
| 1 | Đếm số lần mỗi số xuất hiện | Số xuất hiện 1 lần là đầu mút | Bậc: 10:2, 20:1, 30:2, 40:1 |
| 2 | `xr[x]` = XOR các hàng xóm | `xr[7] = -1 ^ 0`, `xr[-1] = 7`, `xr[0] = 7` | Đỉnh bậc 1 đầu tiên được chèn là `-1` |
| 3 | `Counter` hai cột | lọc `c == 1` | `tuple(sorted(...))` |
| 4 | `g.setdefault(a, []).append(b)` | Phần còn lại y như Cách 3 | Nhớ cả chiều `b → a` |
| 5 | Mọi đỉnh bậc 2 | Bắt đầu `m = min`, `ans = [m, min(g[m])]` | Lặp "hàng xóm khác prev" cho đủ `n` phần tử |

## 19. Đáp án tham khảo

In [32]:
# Bài 1
p1 = [[10, 20], [30, 10], [40, 30]]
deg1 = Counter(x for q in p1 for x in q)
print("Bài 1:", dict(deg1), "đầu mút:", sorted(x for x, d in deg1.items() if d == 1))
assert sorted(x for x, d in deg1.items() if d == 1) == [20, 40]

# Bài 2
p2 = [[7, -1], [0, 7]]
r2 = trace_xor(p2)
assert r2 == [-1, 7, 0]
print("Bài 2:", r2)

Bài 1: {10: 2, 20: 1, 30: 2, 40: 1} đầu mút: [20, 40]
xr = {7: -1, -1: 7, 0: 7} | deg = {7: 2, -1: 1, 0: 1}
bắt đầu -1, hàng xóm duy nhất xr[-1]=7
cur=7, prev=-1: xr[cur]=-1 ^ -1 = 0
Bài 2: [-1, 7, 0]


In [33]:
def find_endpoints(pairs):
    deg = Counter(x for q in pairs for x in q)
    return tuple(sorted(x for x, d in deg.items() if d == 1))

def restore_plain_dict(pairs):
    g = {}
    for a, b in pairs:
        g.setdefault(a, []).append(b)
        g.setdefault(b, []).append(a)
    start = next(x for x, v in g.items() if len(v) == 1)
    ans = [start, g[start][0]]
    while len(ans) < len(pairs) + 1:
        v = g[ans[-1]]
        ans.append(v[1] if v[0] == ans[-2] else v[0])
    return ans

assert find_endpoints([[2, 1], [3, 4], [3, 2]]) == (1, 4)
assert find_endpoints([[4, -2], [1, 4], [-3, 1]]) == (-3, -2)
rng = random.Random(5)
for _ in range(200):
    n = rng.randint(2, 50)
    arr, pairs = gen_case(n, rng)
    assert find_endpoints(pairs) == tuple(sorted((arr[0], arr[-1])))
    r = restore_plain_dict(pairs)
    assert is_valid(pairs, r)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [34]:
def restore_cycle(pairs):
    g = defaultdict(list)
    for a, b in pairs:
        g[a].append(b); g[b].append(a)
    n = len(pairs)
    m = min(g)
    ans = [m, min(g[m])]
    while len(ans) < n:
        v = g[ans[-1]]
        ans.append(v[1] if v[0] == ans[-2] else v[0])
    return ans

def cycle_brute(pairs):
    # chuẩn: thử mọi hoán vị, giữ hoán vị hợp lệ dạng vòng, chọn bản chuẩn hóa
    vals = sorted({x for q in pairs for x in q})
    want = Counter(frozenset(q) for q in pairs)
    best = []
    for perm in permutations(vals):
        k = len(perm)
        got = Counter(frozenset((perm[i], perm[(i + 1) % k])) for i in range(k))
        if got == want and perm[0] == vals[0] and perm[1] < perm[-1]:
            best.append(list(perm))
    assert len(best) == 1
    return best[0]

print("ví dụ tự dựng (đối chiếu lại đề gốc — không thuộc đề):", restore_cycle([[3, 1], [1, 4], [4, 2], [2, 3]]))
rng = random.Random(9)
for _ in range(150):
    n = rng.randint(3, 7)
    arr = rng.sample(range(-20, 21), n)
    pairs = [[arr[i], arr[(i + 1) % n]] for i in range(n)]
    rng.shuffle(pairs)
    assert restore_cycle(pairs) == cycle_brute(pairs)
print("Bài 5: OK (đối chiếu brute force)")

ví dụ tự dựng (đối chiếu lại đề gốc — không thuộc đề): [1, 3, 2, 4]


Bài 5: OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
g = defaultdict(list)
for a, b in pairs:
    g[a].append(b); g[b].append(a)          # vô hướng: thêm 2 chiều
start = next(x for x, v in g.items() if len(v) == 1)   # đầu mút
ans = [start, g[start][0]]
while len(ans) < len(pairs) + 1:
    v = g[ans[-1]]
    ans.append(v[1] if v[0] == ans[-2] else v[0])     # hàng xóm khác prev
```

**Checklist:**
- [ ] Đã thêm cạnh **cả hai chiều**?
- [ ] Bắt đầu từ đỉnh **bậc 1**, không phải `pairs[0][0]`?
- [ ] Trường hợp `n = 2` (một cặp) chạy đúng? (vòng lặp `range(2, n)` rỗng)
- [ ] Không dùng `0`/`-1` làm cha giả; không DFS đệ quy sâu `10⁵`?
- [ ] Lưu hàng xóm bằng list (hoặc XOR), không ghi đè `g[a] = b`?
- [ ] Độ phức tạp `O(n)`; trả về list độ dài `len(pairs) + 1`.